In [ ]:
from collections import namedtuple
import math
import functools
cache = functools.lru_cache(10**6)

In [ ]:
class Game:
    """A game is similar to a problem, but it has a terminal test instead of 
    a goal test, and a utility for each terminal state. To create a game, 
    subclass this class and implement `actions`, `result`, `is_terminal`, 
    and `utility`. You will also need to set the .initial attribute to the 
    initial state; this can be done in the constructor."""

    def actions(self, state):
        """Return a collection of the allowable moves from this state."""
        raise NotImplementedError

    def result(self, state, move):
        """Return the state that results from making a move from a state."""
        raise NotImplementedError

    def is_terminal(self, state):
        """Return True if this is a final state for the game."""
        return not self.actions(state)
    
    def utility(self, state, player):
        """Return the value of this final state to player."""
        raise NotImplementedError

    def terminal_test(self, state):
        return self.is_terminal(state)

    def to_move(self,state):
        return state.to_move


        

def play_game(game, strategies: dict, verbose=False):
    """Play a turn-taking game. `strategies` is a {player_name: function} dict,
    where function(state, game) is used to get the player's move."""
    state = game.initial
    while not game.is_terminal(state):
        player = state.to_move
        move = strategies[player](game, state)
        state = game.result(state, move)
        if verbose: 
            print('Player', player, 'move:', move)
            print(state)
        state = game.add_random_barrier(state)
        print(state)
    return state

In [ ]:
def minimax_search(game, state):
    """Search game tree to determine best move; return (value, move) pair."""

    player = state.to_move

    def max_value(state):
        if game.is_terminal(state):
            return game.utility(state, player), None
        v, move = -infinity, None
        for a in game.actions(state):
            v2, _ = min_value(game.result(state, a))
            if v2 > v:
                v, move = v2, a
        return v, move

    def min_value(state):
        if game.is_terminal(state):
            return game.utility(state, player), None
        v, move = +infinity, None
        for a in game.actions(state):
            v2, _ = max_value(game.result(state, a))
            if v2 < v:
                v, move = v2, a
        return v, move

    return max_value(state)

infinity = math.inf

def alphabeta_search(game, state):
    """Search game to determine best action; use alpha-beta pruning.
    As in [Figure 5.7], this version searches all the way to the leaves."""

    player = state.to_move

    def max_value(state, alpha, beta):
        if game.is_terminal(state):
            return game.utility(state, player), None
        v, move = -infinity, None
        for a in game.actions(state):
            v2, _ = min_value(game.result(state, a), alpha, beta)
            if v2 > v:
                v, move = v2, a
                alpha = max(alpha, v)
            if v >= beta:
                return v, move
        return v, move

    def min_value(state, alpha, beta):
        if game.is_terminal(state):
            return game.utility(state, player), None
        v, move = +infinity, None
        for a in game.actions(state):
            v2, _ = max_value(game.result(state, a), alpha, beta)
            if v2 < v:
                v, move = v2, a
                beta = min(beta, v)
            if v <= alpha:
                return v, move
        return v, move

    return max_value(state, -infinity, +infinity)

In [ ]:
def minimax_search_tt(game, state):
    """Search game to determine best move; return (value, move) pair."""

    player = state.to_move

    @cache
    def max_value(state):
        if game.is_terminal(state):
            return game.utility(state, player), None
        v, move = -infinity, None
        for a in game.actions(state):
            v2, _ = min_value(game.result(state, a))
            if v2 > v:
                v, move = v2, a
        return v, move

    @cache
    def min_value(state):
        if game.is_terminal(state):
            return game.utility(state, player), None
        v, move = +infinity, None
        for a in game.actions(state):
            v2, _ = max_value(game.result(state, a))
            if v2 < v:
                v, move = v2, a
        return v, move

    return max_value(state)

In [ ]:
from collections import defaultdict

class Board(defaultdict):
    """A board has the player to move, a cached utility value, 
    and a dict of {(x, y): player} entries, where player is 'X' or 'O'."""
    empty = '.'
    off = '#'
    
    def __init__(self, width=8, height=8, to_move=None, **kwds):
        self.__dict__.update(width=width, height=height, to_move=to_move, **kwds)
        
    def new(self, changes: dict, **kwds) -> 'Board':
        "Given a dict of {(x, y): contents} changes, return a new Board with the changes."
        board = Board(width=self.width, height=self.height, **kwds)
        board.update(self)
        board.update(changes)
        return board

    def __missing__(self, loc):
        x, y = loc
        if 0 <= x < self.width and 0 <= y < self.height:
            return self.empty
        else:
            return self.off
            
    def __hash__(self): 
        return hash(tuple(sorted(self.items()))) + hash(self.to_move)
    
    def __repr__(self):
        def row(y): return ' '.join(self[x, y] for x in range(self.width))
        return '\n'.join(map(row, range(self.height))) +  '\n'

In [ ]:
import random
class TicTacToe(Game):
    """Play TicTacToe on an `height` by `width` board, needing `k` in a row to win.
    'X' plays first against 'O'."""

    def __init__(self, height=5, width=5, k=5):
        self.k = k # k in a row
        self.squares = {(x, y) for x in range(width) for y in range(height)}
        self.initial = Board(height=height, width=width, to_move='X', utility=0)

    def actions(self, board):
        """Legal moves are any square not yet taken."""
        return self.squares - set(board)

    def result(self, board, square):
        """Place a marker for current player on square."""
        player = board.to_move
        board = board.new({square: player}, to_move=('O' if player == 'X' else 'X'))
        win = k_in_row(board, player, square, self.k)
        board.utility = (0 if not win else +1 if player == 'X' else -1)
        return board

    def utility(self, board, player):
        """Return the value to player; 1 for win, -1 for loss, 0 otherwise."""
        return board.utility if player == 'X' else -board.utility

    def is_terminal(self, board):
        """A board is a terminal state if it is won or there are no empty squares."""
        return board.utility != 0 or len(self.squares) == len(board)

    def display(self, board): print(board)     

    def add_random_barrier(self, board):
        """Add a random barrier to the board."""
        player = board.to_move
        square = random.choice(list(self.actions(board))) if self.actions(board) else None
        if square is None:
            return board
        print('adding random barrier at', square)
        board = board.new({square: '#'}, to_move=board.to_move)
        win = k_in_row(board, player, square, self.k)
        board.utility = (0 if not win else +1 if player == 'X' else -1)
        return board


def k_in_row(board, player, square, k):
    """True if player has k pieces in a line through square."""
    def in_row(x, y, dx, dy): return 0 if board[x, y] != player else 1 + in_row(x + dx, y + dy, dx, dy)
    return any(in_row(*square, dx, dy) + in_row(*square, -dx, -dy) - 1 >= k
               for (dx, dy) in ((0, 1), (1, 0), (1, 1), (1, -1)))

def __repr__(self):
    return self.__class__.__name__ + ' ' + str(dict(self))    

In [ ]:
import random

def random_player(game, state): return random.choice(list(game.actions(state)))

def player(search_algorithm):
    """A game player who uses the specified search algorithm"""
    return lambda game, state: search_algorithm(game, state)[1]

In [ ]:
def cache1(function):
    "Like lru_cache(None), but only considers the first argument of function."
    cache = {}
    def wrapped(x, *args):
        if x not in cache:
            cache[x] = function(x, *args)
        return cache[x]
    return wrapped

def alphabeta_search_tt(game, state):
    """Search game to determine best action; use alpha-beta pruning.
    As in [Figure 5.7], this version searches all the way to the leaves."""

    player = state.to_move

    @cache1
    def max_value(state, alpha, beta):
        if game.is_terminal(state):
            return game.utility(state, player), None
        v, move = -infinity, None
        for a in game.actions(state):
            v2, _ = min_value(game.result(state, a), alpha, beta)
            if v2 > v:
                v, move = v2, a
                alpha = max(alpha, v)
            if v >= beta:
                return v, move
        return v, move

    @cache1
    def min_value(state, alpha, beta):
        if game.is_terminal(state):
            return game.utility(state, player), None
        v, move = +infinity, None
        for a in game.actions(state):
            v2, _ = max_value(game.result(state, a), alpha, beta)
            if v2 < v:
                v, move = v2, a
                beta = min(beta, v)
            if v <= alpha:
                return v, move
        return v, move

    return max_value(state, -infinity, +infinity)

In [ ]:
def cutoff_depth(d):
    """A cutoff function that searches to depth d."""
    return lambda game, state, depth: depth > d

def h_alphabeta_search(game, state, cutoff=cutoff_depth(7), h=lambda s, p: 0):
    """Search game to determine best action; use alpha-beta pruning.
    As in [Figure 5.7], this version searches all the way to the leaves."""
    print('__________')
    print('game', game.__dict__)
    print('__________')
    print('state', state.__dict__)
    print('__________')

    player = state.to_move

   # @cache1
    def max_value(state, alpha, beta, depth):
        if game.is_terminal(state):
            return game.utility(state, player), None
        if cutoff(game, state, depth):
            return h(state, player), None
        v, move = -infinity, None
        for a in game.actions(state):
            v2, _ = min_value(game.result(state, a), alpha, beta, depth+1)
            if v2 > v:
                v, move = v2, a
                alpha = max(alpha, v)
            if v >= beta:
                return v, move
        return v, move

   # @cache1
    def min_value(state, alpha, beta, depth):
        if game.is_terminal(state):
            return game.utility(state, player), None
        if cutoff(game, state, depth):
            return h(state, player), None
        v, move = +infinity, None
        for a in game.actions(state):
            v2, _ = max_value(game.result(state, a), alpha, beta, depth + 1)
            if v2 < v:
                v, move = v2, a
                beta = min(beta, v)
            if v <= alpha:
                return v, move
        return v, move

    return max_value(state, -infinity, +infinity, 0)

play_game(TicTacToe(), dict(X=player(h_alphabeta_search), O=player(h_alphabeta_search)), verbose=True)

In [ ]:

play_game(TicTacToe(height=3, width=3, k=3), dict(X=player(h_alphabeta_search), O=player(h_alphabeta_search)), verbose=True)

In [ ]:
#minmax search using tensor flow
import tensorflow as tf
import numpy as np

def minimax_search_tf(game, state):
    """Search game tree to determine best move; return (value, move) pair."""
    
    player = state.to_move

    def max_value(state):
        if game.is_terminal(state):
            return game.utility(state, player), None
        v, move = -infinity, None
        for a in game.actions(state):
            v2, _ = min_value(game.result(state, a))
            if v2 > v:
                v, move = v2, a
        return v, move

    def min_value(state):
        if game.is_terminal(state):
            return game.utility(state, player), None
        v, move = +infinity, None
        for a in game.actions(state):
            v2, _ = max_value(game.result(state, a))
            if v2 < v:
                v, move = v2, a
        return v, move

    return max_value(state)

In [ ]:
%pip install tensorflow
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import random

In [ ]:
# Build Neural Network Model
def create_model():
    model = keras.Sequential([
        layers.Dense(128, activation='relu', input_shape=(25,)),
        layers.Dense(128, activation='relu'),
        layers.Dense(25, activation='linear')  # Output layer for Q-values of each move
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mse')
    return model

# Train the model
def train_model(model, episodes=10000, gamma=0.9, epsilon=0.1):
    game = TicTacToe()
    for episode in range(episodes):
        state = game.reset()
        done = False
        while not done:
            if random.uniform(0, 1) < epsilon:  # Explore
                move = random.choice(game.available_moves())
            else:  # Exploit
                q_values = model.predict(state.reshape(1, 25), verbose=0)
                move_index = np.argmax(q_values)
                move = (move_index // 5, move_index % 5)
            
            if game.make_move(*move):
                next_state = game.board.flatten()
                reward = 0
                winner = game.check_winner()
                if winner == 1:
                    reward = 1  # Win reward
                    done = True
                elif winner == 2:
                    reward = -1  # Lose penalty
                    done = True
                elif winner == 0:
                    reward = 0.5  # Draw
                    done = True
                
                target = model.predict(state.reshape(1, 25), verbose=0)
                target[0][move[0] * 5 + move[1]] = reward + gamma * np.max(model.predict(next_state.reshape(1, 25), verbose=0))
                
                model.fit(state.reshape(1, 25), target, epochs=1, verbose=0)
                state = next_state
        
        if episode % 1000 == 0:
            print(f"Episode {episode}/{episodes}")

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import random

# Tic-Tac-Toe board representation
class TicTacToe:
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.board = np.zeros((5, 5), dtype=int)
        self.current_player = 1  # Player 1 starts
        return self.board.flatten()
    
    def available_moves(self):
        return [(r, c) for r in range(5) for c in range(5) if self.board[r, c] == 0]
    
    def make_move(self, row, col):
        if self.board[row, col] == 0:
            self.board[row, col] = self.current_player
            self.current_player = 3 - self.current_player  # Switch players
            return True
        return False
    
    def check_winner(self):
        for player in [1, 2]:
            for row in range(5):
                if np.all(self.board[row, :] == player):
                    return player
            for col in range(5):
                if np.all(self.board[:, col] == player):
                    return player
            if np.all(np.diag(self.board) == player) or np.all(np.diag(np.fliplr(self.board)) == player):
                return player
        if len(self.available_moves()) == 0:
            return 0  # Draw
        return None  # Game not finished

# Build Neural Network Model
def create_model():
    model = keras.Sequential([
        layers.Dense(128, activation='relu', input_shape=(25,)),
        layers.Dense(128, activation='relu'),
        layers.Dense(25, activation='linear')  # Output layer for Q-values of each move
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mse')
    return model

# Train the model
def train_model(model, episodes=10000, gamma=0.9, epsilon=0.1):
    game = TicTacToe()
    for episode in range(episodes):
        state = game.reset()
        done = False
        while not done:
            if random.uniform(0, 1) < epsilon:  # Explore
                move = random.choice(game.available_moves())
            else:  # Exploit
                q_values = model.predict(state.reshape(1, 25), verbose=0)
                move_index = np.argmax(q_values)
                move = (move_index // 5, move_index % 5)
            
            if game.make_move(*move):
                next_state = game.board.flatten()
                reward = 0
                winner = game.check_winner()
                if winner == 1:
                    reward = 1  # Win reward
                    done = True
                elif winner == 2:
                    reward = -1  # Lose penalty
                    done = True
                elif winner == 0:
                    reward = 0.5  # Draw
                    done = True
                
                target = model.predict(state.reshape(1, 25), verbose=0)
                target[0][move[0] * 5 + move[1]] = reward + gamma * np.max(model.predict(next_state.reshape(1, 25), verbose=0))
                
                model.fit(state.reshape(1, 25), target, epochs=1, verbose=0)
                state = next_state
        
        if episode % 1000 == 0:
            print(f"Episode {episode}/{episodes}")

# Let a human play against the trained AI
def play_against_ai(model):
    game = TicTacToe()
    while True:
        print(game.board)
        if game.current_player == 1:
            row, col = map(int, input("Enter row and column (0-4): ").split())
            if not game.make_move(row, col):
                print("Invalid move, try again.")
                continue
        else:
            q_values = model.predict(game.board.flatten().reshape(1, 25), verbose=0)
            move_index = np.argmax(q_values)
            move = (move_index // 5, move_index % 5)
            game.make_move(*move)
            print(f"AI plays: {move}")
        
        winner = game.check_winner()
        if winner is not None:
            print(game.board)
            if winner == 1:
                print("You win!")
            elif winner == 2:
                print("AI wins!")
            else:
                print("It's a draw!")
            break

# Create and train the model
model = create_model()
train_model(model, episodes=5000)
play_against_ai(model)


In [ ]:
def minimaxFlip(self, game, depth, is_maximizing):
        """Minimax recursive function with move and flip consideration."""
        score = self.evaluate(game)
        if score != 0 or len(game.available_moves()) == 0:
            return score

        if is_maximizing:
            best_score = -np.inf
            for move in game.available_moves():
                game.make_move(*move)
                score = self.minimax(game, depth + 1, False)
                game.board[move] = 0  # Undo move
                game.current_player = 1
                best_score = max(best_score, score)

            if not game.flip_used[1]:  # Consider flip if not used
                for flip in game.available_flips():
                    game.flip_mark(*flip)
                    score = self.minimax(game, depth + 1, False)
                    game.board[flip] = 2  # Undo flip
                    game.flip_used[1] = False
                    best_score = max(best_score, score)

            return best_score

        else:
            best_score = np.inf
            for move in game.available_moves():
                game.make_move(*move)
                score = self.minimax(game, depth + 1, True)
                game.board[move] = 0  # Undo move
                game.current_player = 2
                best_score = min(best_score, score)

            if not game.flip_used[2]:  # Consider flip if not used
                for flip in game.available_flips():
                    game.flip_mark(*flip)
                    score = self.minimax(game, depth + 1, True)
                    game.board[flip] = 1  # Undo flip
                    game.flip_used[2] = False
                    best_score = min(best_score, score)

            return best_score

def best_move(self, game):
        """Find the best move using Minimax."""
        best_score = -np.inf
        best_move = None

        for move in game.available_moves():
            game.make_move(*move)
            score = self.minimax(game, 0, False)
            game.board[move] = 0  # Undo move
            game.current_player = 1
            if score > best_score:
                best_score = score
                best_move = ("M", move)

        if not game.flip_used[1]:
            for flip in game.available_flips():
                game.flip_mark(*flip)
                score = self.minimax(game, 0, False)
                game.board[flip] = 2  # Undo flip
                game.flip_used[1] = False
                if score > best_score:
                    best_score = score
                    best_move = ("F", flip)

        return best_move



In [ ]:
def minimaxDelayFlip(self, game, depth, is_maximizing):
        """Minimax recursive function with move and late-game flip consideration."""
        score = self.evaluate(game)
        if score != 0 or len(game.available_moves()) == 0:
            return score

        if is_maximizing:
            best_score = -np.inf
            for move in game.available_moves():
                game.make_move(*move)
                score = self.minimax(game, depth + 1, False)
                game.board[move] = 0  # Undo move
                game.current_player = 1
                best_score = max(best_score, score)

            # Hold the flip until the board has less than 3 empty spots
            if not game.flip_used[1] and len(game.available_moves()) < 3:
                for flip in game.available_flips():
                    game.flip_mark(*flip)
                    score = self.minimax(game, depth + 1, False)
                    game.board[flip] = 2  # Undo flip
                    game.flip_used[1] = False
                    best_score = max(best_score, score)

            return best_score

        else:
            best_score = np.inf
            for move in game.available_moves():
                game.make_move(*move)
                score = self.minimax(game, depth + 1, True)
                game.board[move] = 0  # Undo move
                game.current_player = 2
                best_score = min(best_score, score)

            if not game.flip_used[2] and len(game.available_moves()) < 3:
                for flip in game.available_flips():
                    game.flip_mark(*flip)
                    score = self.minimax(game, depth + 1, True)
                    game.board[flip] = 1  # Undo flip
                    game.flip_used[2] = False
                    best_score = min(best_score, score)

            return best_score

In [ ]:
def winning_probability(self, game):
        """Estimate AI's winning probability using Monte Carlo simulations."""
        wins = 0
        for _ in range(self.simulations):
            if game.simulate_random_game() == 1:
                wins += 1
        return wins / self.simulations

def minimax_win_pb(self, game, depth, is_maximizing):
        """Minimax recursive function."""
        score = self.evaluate(game)
        if score != 0 or len(game.available_moves()) == 0:
            return score

        if is_maximizing:
            best_score = -np.inf
            for move in game.available_moves():
                game.make_move(*move)
                score = self.minimax(game, depth + 1, False)
                game.board[move] = 0  # Undo move
                game.current_player = 1
                best_score = max(best_score, score)

            return best_score
        else:
            best_score = np.inf
            for move in game.available_moves():
                game.make_move(*move)
                score = self.minimax(game, depth + 1, True)
                game.board[move] = 0  # Undo move
                game.current_player = 2
                best_score = min(best_score, score)

            return best_score

def best_move(self, game):
        """Find the best move using Minimax and apply flips if probability threshold is met."""
        best_score = -np.inf
        best_move = None

        # Calculate AI's initial winning probability
        initial_win_prob = self.winning_probability(game)
        print(f"AI initial winning probability: {initial_win_prob:.2f}")

        # Consider all normal moves
        for move in game.available_moves():
            game.make_move(*move)
            score = self.minimax(game, 0, False)
            game.board[move] = 0  # Undo move
            game.current_player = 1
            if score > best_score:
                best_score = score
                best_move = ("M", move)

        # Consider flipping only if probability threshold is met
        if not game.flip_used[1]:
            for flip in game.available_flips():
                game.flip_mark(*flip)
                new_win_prob = self.winning_probability(game)
                game.board[flip] = 2  # Undo flip
                game.flip_used[1] = False

                print(f"After flipping {flip}, AI winning probability: {new_win_prob:.2f}")

                # Flip if probability jumps over 90% or drops below 10%
                if new_win_prob >= 0.9 or new_win_prob <= 0.1:
                    best_move = ("F", flip)
                    print(f"AI decides to flip at {flip}!")

        return best_move